# Tool Integration and API Connectivity

## Overview

This notebook demonstrates how to integrate external tools, APIs, and functions into agentic AI systems, extending agent capabilities beyond language generation.

## Learning Objectives

- Understand tool integration patterns for agents
- Build custom tools with proper error handling
- Integrate REST APIs with retry logic
- Implement multimodal tools (vision, audio)
- Apply best practices for production tool integration

## Exam Objectives: 2.2, 2.3, 2.4
## Estimated Time: 90-105 minutes

## 🎯 Exam Focus

This notebook covers concepts that appear frequently on the NCP-AAI exam:

**High-Priority Topics:**
- ⭐⭐⭐ Error handling patterns (retry logic, circuit breakers)
- ⭐⭐⭐ Prompt engineering techniques
- ⭐⭐ Tool integration best practices
- ⭐⭐ Streaming responses

**Common Exam Scenarios:**
- Implementing retry logic for API failures
- Designing prompts for complex tasks
- Handling tool execution errors

**Key Concepts to Master:**
- Exponential backoff for retries
- Circuit breaker pattern
- Dynamic prompt chains
- Graceful degradation

## Setup: Import Dependencies

In [ ]:
# Import core libraries for tool integration
import os
import sys
import time
import logging
from typing import Dict, List, Any, Optional, Callable
from datetime import datetime
from functools import wraps

# LangChain imports for tool creation
from langchain.tools import Tool, BaseTool
from langchain.pydantic_v1 import BaseModel, Field

# Setup logging for debugging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Verify LangChain is available
try:
    import langchain
    print(f"✅ LangChain version: {langchain.__version__}")
except ImportError as e:
    print(f"❌ LangChain not available: {e}")
    print("   Install with: pip install langchain")

# Check for requests library (for API calls)
try:
    import requests
    print("✅ Requests library available")
except ImportError:
    print("⚠️  Requests not available")
    print("   Install with: pip install requests")

print("\n🎯 Setup complete!")

## Theory: Tool Integration Patterns

### What are Tools?

**Tools** extend agent capabilities by connecting to external systems, APIs, and functions. They enable agents to:
- Retrieve real-time information (search, databases)
- Perform computations (calculations, code execution)
- Take actions (send emails, create tickets)
- Process multimodal data (images, audio)

### Tool Categories

| Category | Examples | Use Cases |
|----------|----------|----------|
| **Search** | Web search, document search | Information retrieval |
| **Computation** | Calculator, code execution | Mathematical operations |
| **Data Access** | Database queries, API calls | Retrieve structured data |
| **Actions** | Send email, create ticket | Perform operations |
| **Multimodal** | Image analysis, speech-to-text | Handle non-text data |

### Tool Integration Architecture

```
Agent → Tool Selection → Tool Execution → Result Processing → Response
         ↓                    ↓                  ↓
    LLM decides        API call with       Parse and validate
    which tool         error handling      tool output
```

### Key Considerations

1. **Error Handling**: Tools must handle failures gracefully
2. **Timeouts**: Prevent tools from blocking indefinitely
3. **Retry Logic**: Handle transient failures
4. **Rate Limiting**: Respect API limits
5. **Security**: Validate inputs, sanitize outputs

## Implementation: Basic Tool Creation

Let's create simple tools using LangChain's Tool interface.

In [ ]:
# Method 1: Simple function wrapper
def calculate_sum(numbers: str) -> str:
    """
    Calculate sum of comma-separated numbers.
    
    Args:
        numbers: Comma-separated numbers (e.g., "1,2,3,4")
    
    Returns:
        Sum as string
    """
    try:
        # Parse input string into list of numbers
        num_list = [float(n.strip()) for n in numbers.split(',')]
        # Calculate sum
        result = sum(num_list)
        return f"Sum: {result}"
    except ValueError as e:
        # Handle invalid input gracefully
        return f"Error: Invalid number format. Please provide comma-separated numbers."
    except Exception as e:
        # Catch any other errors
        return f"Error: {str(e)}"

# Create tool from function
calculator_tool = Tool(
    name="calculator",
    func=calculate_sum,
    description="Calculate sum of comma-separated numbers. Input: '1,2,3,4'"
)

# Test the tool
print("=== Testing Calculator Tool ===")
print(f"Input: '10,20,30,40'")
print(f"Output: {calculator_tool.run('10,20,30,40')}")
print(f"\nInput: 'invalid'")
print(f"Output: {calculator_tool.run('invalid')}")

# Method 2: Tool with metadata
def search_documents(query: str) -> str:
    """
    Search internal documentation (simulated).
    
    Args:
        query: Search query
    
    Returns:
        Search results
    """
    # Simulate document search
    mock_docs = {
        "rag": "RAG (Retrieval-Augmented Generation) combines retrieval with generation.",
        "agent": "Agents are autonomous systems that perceive, reason, and act.",
        "llm": "Large Language Models are trained on vast text corpora."
    }
    
    # Find matching documents
    query_lower = query.lower()
    results = [doc for key, doc in mock_docs.items() if key in query_lower]
    
    if results:
        return f"Found {len(results)} document(s):\n" + "\n".join(results)
    else:
        return "No documents found matching your query."

search_tool = Tool(
    name="search_docs",
    func=search_documents,
    description="Search internal documentation. Input: search query string"
)

# Test search tool
print("\n=== Testing Search Tool ===")
print(f"Query: 'RAG'")
print(f"Result: {search_tool.run('RAG')}")
print(f"\nQuery: 'unknown topic'")
print(f"Result: {search_tool.run('unknown topic')}")

## Implementation: Structured Tools with Pydantic

For more complex tools, use Pydantic for input validation.

In [ ]:
# Define input schema with Pydantic
class WeatherInput(BaseModel):
    """Input schema for weather tool."""
    location: str = Field(description="City name or zip code")
    units: str = Field(default="fahrenheit", description="Temperature units: fahrenheit or celsius")

class WeatherTool(BaseTool):
    """Tool for getting weather information."""
    
    name = "get_weather"
    description = "Get current weather conditions for a location. Provide city name and optional units."
    args_schema = WeatherInput
    
    def _run(self, location: str, units: str = "fahrenheit") -> str:
        """
        Execute the weather tool.
        
        Args:
            location: City name or zip code
            units: Temperature units
        
        Returns:
            Weather information string
        """
        try:
            # Simulate weather API call
            mock_weather = {
                "san francisco": {"temp": 65, "conditions": "Partly cloudy"},
                "new york": {"temp": 72, "conditions": "Sunny"},
                "london": {"temp": 58, "conditions": "Rainy"}
            }
            
            # Lookup weather data
            location_lower = location.lower()
            if location_lower in mock_weather:
                data = mock_weather[location_lower]
                temp = data['temp']
                
                # Convert to celsius if requested
                if units.lower() == "celsius":
                    temp = round((temp - 32) * 5/9, 1)
                    unit_symbol = "°C"
                else:
                    unit_symbol = "°F"
                
                return f"Weather in {location}: {temp}{unit_symbol}, {data['conditions']}"
            else:
                return f"Weather data not available for {location}"
                
        except Exception as e:
            # Handle errors gracefully
            logger.error(f"Weather tool error: {str(e)}")
            return f"Error retrieving weather: {str(e)}"
    
    async def _arun(self, location: str, units: str = "fahrenheit") -> str:
        """
        Async version of the tool.
        
        Note: Not implemented for this demo
        """
        raise NotImplementedError("Async version not implemented")

# Create and test weather tool
weather_tool = WeatherTool()

print("=== Testing Structured Weather Tool ===")
print(f"Query: San Francisco (Fahrenheit)")
print(f"Result: {weather_tool.run({'location': 'San Francisco', 'units': 'fahrenheit'})}")

print(f"\nQuery: London (Celsius)")
print(f"Result: {weather_tool.run({'location': 'London', 'units': 'celsius'})}")

print(f"\nQuery: Unknown City")
print(f"Result: {weather_tool.run({'location': 'Unknown City', 'units': 'fahrenheit'})}")

## References

### Course Materials

**Course Notes:** [Module 2: Agent Development](../../course-notes/module-02-agent-development.md)

### Practice Questions

**Exam Questions:** [Practice Questions](../../exam-questions/domain-02-development.md)

### Hands-On Labs

- [Lab 1: Basic RAG Agent](../../labs/lab-01-basic-rag-agent/README.md)
- [Lab 2: Multi-Agent Research](../../labs/lab-02-multi-agent-research/README.md)

### Additional Resources

- [NVIDIA NeMo Documentation](https://docs.nvidia.com/nemo/)
- [LangChain Documentation](https://python.langchain.com/)
- [NVIDIA AI Endpoints](https://build.nvidia.com/)
